In [ ]:
import sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
sys.path.append("..")
from src.data_generation.SSVI import sample_params, ssvi
from src.evaluation.surface_eval import check_arbitrage

cfg = yaml.safe_load(open("../config.yaml"))

In [ ]:
np.random.seed(0)
ttms = np.geomspace(cfg["ttm"]["min"], cfg["ttm"]["max"], cfg["ttm"]["n_points"])
ks = np.linspace(cfg["k"]["min"], cfg["k"]["max"], cfg["k"]["n_points"])

rho, eta, gamma, v_bar, v0, kappa = sample_params(cfg, n=1000)

surfaces = ssvi(ttms, ks, rho, eta, gamma, v_bar, v0, kappa) 

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), subplot_kw={"projection": "3d"})
K, T = np.meshgrid(ks, ttms)

for i, ax in enumerate(axes.flat):
    iv = surfaces[i]
    ax.plot_surface(K, T, iv, cmap="viridis", edgecolor="none", alpha=0.9)
    ax.set_xlabel("k")
    ax.set_ylabel("tau")
    ax.set_zlabel("sigma")
    label = (
        f"rho={rho[i]:.2f}  eta={eta[i]:.2f}  gamma={gamma[i]:.2f} \n "
        f"vbar={v_bar[i]:.2f}  v0={v0[i]:.2f}  kappa={kappa[i]:.2f}"
    )
    ax.text2D(0.5, -0.05, label, transform=ax.transAxes, ha="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
cal_violations, butterfly_violations = check_arbitrage(surfaces, ttms, ks)
print(f"Calendar spread violations: {cal_violations.mean()*100}% ({cal_violations.sum()}/{len(surfaces)})")
print(f"Butterfly violations:       {butterfly_violations.mean()*100}% ({butterfly_violations.sum()}/{len(surfaces)})")
print(f"Any arbitrage:              {(cal_violations | butterfly_violations).mean()*100}%")